# OpenAI Agent SDK Examples 
### Subject Detection by Triag Agent; Use of tools by Agents, Input Guardrails

The following steps demonstrate:
A **Triage Agent** that detects the subject and routes question to the right subject agent.

### Step 1: Install the OpenAI Agents SDK

In [1]:
#!pip install openai-agents -q

### Step 2: Set the OpenAI API Key

In [6]:
import os
from pprint import pprint
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv, find_dotenv

# Load variables from the .env file into the system environment
_ = load_dotenv(find_dotenv())

# Retrieve the key using standard os.getenv
api_key = os.getenv('OPENAI_API_KEY')

# 3. Verify exactly like you did in Colab
if api_key:
    print(f"API Key loaded successfully! Starts with: {api_key[:7]}...")
else:
    print("API Key not found. Make sure your .env file is in the same folder.")

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

API Key loaded successfully! Starts with: sk-proj...


### Step 3: Import Dependencies

In [7]:
from agents import Agent, Runner
import asyncio

### Step 4: Define two or more subject agents: History Agent and Math Agent and a Triage Agent

In [8]:
history_agent = Agent(
    name="History agent",
    instructions="You answer questions clearly.",
    handoff_description="Specialist agent for history questions",
    model="gpt-4"
)

math_agent = Agent(
    name="Math agent",
    instructions="You only respond to Math questions.",
    handoff_description="Specialist agent for math questions",
    model="gpt-4"
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the subject of the request.",
    handoffs=[history_agent, math_agent],
    model="gpt-4"
)

print("✅ Subject agents and the Triage agent defined.")

✅ Subject agents and the Triage agent defined.


### Step 5: Run the triage agent and one of subject agents

In [9]:
user_input="What is the year of French revolution"
language_result = await Runner.run(triage_agent, input=user_input)
raw_response = language_result.final_output
print(f"\n Subject Agent Response:\n{raw_response}")


 Subject Agent Response:
The French Revolution began in the year 1789.


In [10]:
user_input="What golden ratio?"
language_result = await Runner.run(triage_agent, input=user_input)
raw_response = language_result.final_output
print(f"\n Subject Agent Response:\n{raw_response}")


 Subject Agent Response:
The golden ratio, also known as Phi, is a special number approximately equal to 1.618. It appears many times in geometry, art, architecture and other areas. The golden ratio is the ratio of the length to the width of what is considered to be one of the most aesthetically pleasing rectangular shapes. This rectangle, known as a golden rectangle, can be partitioned into a square and a smaller rectangle that retains the aspect ratio of the original rectangle. The unique feature of the golden ratio is that it can be seen all across the universe, from the shape of the galaxies to the shell of a snail. It's a mathematical pattern that makes objects and works of art more appealing to the human eye.


## Adding a function tool

In [11]:
import asyncio
from agents import Agent, Runner, function_tool

@function_tool
def history_fun_fact() -> str:
    """Return a short history fact."""
    return "Sharks are older than trees."

agent = Agent(
    name="History tutor",
    instructions="Answer history questions clearly. Use history_fun_fact when it helps.",
    tools=[history_fun_fact],
    model="gpt-5.4"
)
user_input="Who is the most famous traitor of the AMerican revolution?"
result = await Runner.run(triage_agent, input=user_input)
raw_response = result.final_output
print(f"\n Response of an agent with a tool:\n{raw_response}")



 Response of an agent with a tool:
The most famous traitor of the American Revolution is Benedict Arnold. He was an American general during the Revolutionary War who defected to the British side.


In [12]:
user_input="Tell me something surprising about ancient life on Earth?"
result = await Runner.run(triage_agent, input=user_input)
raw_response = result.final_output
print(f"\n Response of an agent with a tool:\n{raw_response}")



 Response of an agent with a tool:
One surprising fact about ancient life on Earth is that dinosaurs and flowering plants didn't exist at the same time. Flowering plants (angiosperms) have been ruling the plant kingdom for the last 100 million years, but dinosaurs lived between 230 and 65 million years ago. So, for most of the time dinosaurs roamed the Earth, their environment was dominated by ferns, cycads, ginkgoes and other gymnosperms, instead of flowers. It wasn't until the end of the Cretaceous period that flowering plants started to become prominent!


## Guardrails
Use input guardrails when you want a fast validation step to run before the expensive or side-effecting part of the workflow starts.

In [14]:
import asyncio
from pydantic import BaseModel

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str

guardrail_agent = Agent(
    name="Homework check",
    instructions="Detect whether the user is asking for math homework help.",
    output_type=MathHomeworkOutput,
)

@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )

agent = Agent(
    name="student homework controller",
    instructions="Help students with homework questions.",
    input_guardrails=[math_guardrail],
)

### Let us ask a disallowed Math question

In [15]:
try:
    result = await Runner.run(agent, "Can you solve 2x + 3 = 11 for me?")
    raw_response = result.final_output
    print(f"\n Response of an agent with math quardrails:\n{raw_response}")
except InputGuardrailTripwireTriggered:
    print("Guardrail blocked the request.")

Guardrail blocked the request.


### Now ask a history question

In [16]:
try:
    result = await Runner.run(agent, "When did the World War One end?")
    raw_response = result.final_output
    print(f"\n Response of an agent with math quardrails:\n{raw_response}")
except InputGuardrailTripwireTriggered:
    print("Guardrail blocked the request.")


 Response of an agent with math quardrails:
World War One ended on **November 11, 1918**. This date is known as **Armistice Day**, when the armistice (an agreement to stop fighting) was signed between the Allies and Germany, marking the end of fighting on the Western Front. The official peace treaty, called the Treaty of Versailles, was signed later on **June 28, 1919**.
